In [4]:
import os
from openai import AzureOpenAI
import openai
from openai import OpenAI
import pandas as pd
import spacy
import scispacy
import pandas as pd
import math
import numpy as np
import requests
import re
from collections import Counter

#endpoint = "https://headlamp-open-ai-staging.openai.azure.com/"
#model_name = "gpt-4o"
#deployment = "gpt-4o"

#subscription_key = "673670cdd7684ef28f5996a5747a2ee4"
#api_version = "2024-12-01-preview"

#client = OpenAI(api_key="673670cdd7684ef28f5996a5747a2ee4")



#endpoint = "https://ai-engineering7094ai397789208412.cognitiveservices.azure.com/openai/deployments/o3-deployment/chat/completions?api-version=2025-01-01-preview"
#model_name = "o3"
#deployment = "o3-deployment"

#subscription_key = "66JmQFhvnmR7s0votsddacRNKlCuCSEC1sTCHvPcFnFqNUNEgIAVJQQJ99BAACHYHv6XJ3w3AAAAACOG7sSq"
#api_version = "2025-01-01-preview"

#client = OpenAI(api_key="66JmQFhvnmR7s0votsddacRNKlCuCSEC1sTCHvPcFnFqNUNEgIAVJQQJ99BAACHYHv6XJ3w3AAAAACOG7sSq")



#client = AzureOpenAI(
#    api_version=api_version,
#    azure_endpoint=endpoint,
#    api_key=subscription_key,
#)

#response = client.chat.completions.create(
#    messages=[
#        {
#            "role": "system",
#            "content": "You are a helpful assistant.",
#        },
#        {
#            "role": "user",
#            "content": "I am going to Paris, what should I see?",
#        }
#    ],
#    max_completion_tokens=4096,
#    temperature=1.0,
#    top_p=1.0,
#    model=deployment
#)

#print(response.choices[0].message.content)

In [1]:
import os
from pathlib import Path

# Define the correct folder path
target_folder = Path("/Users/naveendeevi/Python Projects/MDD/MDD_Datasets")

# Check if folder exists
if target_folder.exists() and target_folder.is_dir():
    os.chdir(target_folder)
    print(f"Current working directory: {os.getcwd()}")

    # List all items in the folder
    items = list(target_folder.iterdir())
    if items:
        print("\nItems in directory:")
        for item in items:
            print(f"- {item.name}")
    else:
        print("\nNo items found in the directory.")
else:
    raise FileNotFoundError(f"Folder not found: {target_folder}")

Current working directory: /Users/naveendeevi/Python Projects/MDD/MDD_Datasets

Items in directory:
- openfda_cache_dosage.json
- MDD_Drugs_V1_1.numbers
- Similarity Drugs_V2.numbers
- MDD_Drugs_V1.numbers
- Similarity Drugs_V2.csv
- test.csv
- drug_similarity 2.numbers
- drug_test.csv
- drug_test_copy.csv
- drug_similarity.numbers
- .openfda_pk_cache
- score_tfidf_symantic V3 2.numbers
- score_tfidf_symantic V2.numbers
- MDD_Drugs_V1.csv
- test.numbers
- score_tfidf_symantic V3.numbers
- df_pharmacokinetics.csv
- openfda_cache.json
- openfda_cache_clinical.json


In [5]:
# Functions
#-------------------4o complettion -----------------

#endpoint = "https://headlamp-open-ai-staging.openai.azure.com/"
#model_name = "gpt-4o"
#deployment = "gpt-4o"
#subscription_key = "673670cdd7684ef28f5996a5747a2ee4"
#api_version = "2024-12-01-preview"

#client = AzureOpenAI(
#    api_version=api_version,
#    azure_endpoint=endpoint,
#    api_key=subscription_key,
#)

#def get_completion(prompt, model=model_name):
#    messages = [{"role": "user", "content": prompt}]
#    response = client.chat.completions.create(
#        model=deployment,
#        messages=messages,
#        temperature=0, # this is the degree of randomness of the model's output
#    )
#    return response.choices[0].message.content


#-----------------O3 completion --------------

endpoint = "https://ai-engineering7094ai397789208412.cognitiveservices.azure.com/openai/deployments/o3-deployment/chat/completions?api-version=2025-01-01-preview"
model_name = "o3"
deployment = "o3-deployment"
subscription_key = "66JmQFhvnmR7s0votsddacRNKlCuCSEC1sTCHvPcFnFqNUNEgIAVJQQJ99BAACHYHv6XJ3w3AAAAACOG7sSq"
api_version = "2025-01-01-preview"
client = OpenAI(api_key="66JmQFhvnmR7s0votsddacRNKlCuCSEC1sTCHvPcFnFqNUNEgIAVJQQJ99BAACHYHv6XJ3w3AAAAACOG7sSq")


client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=subscription_key,
    api_version=api_version,
)

def get_completion_o3(prompt, deployment=deployment):
    messages = [{"role": "user", "content": prompt}]
    resp = client.chat.completions.create(
        model=deployment,                 # <-- your Azure deployment name
        messages=messages,
        #max_completion_tokens=300,        # o-series use this field
        # no temperature/top_p for o3
    )
    return resp.choices[0].message.content.strip()

def build_adme_prompt(text: str) -> str:
    return f"""
You are extracting ADME (absorption, distribution, metabolism, excretion) facts from the 'Pharmacokinetics' text.

Rules:
- Use ONLY the provided text. Do not add outside info.
- Return concise bullet points (≤ 25 words each), directly supported by the text.
- If a category has no info, return an empty array [].
- Output JSON ONLY with this exact schema and key order:

{{
  "absorption": ["..."],
  "distribution": ["..."],
  "metabolism": ["..."],
  "excretion": ["..."]
}}

Text:
```{text}```
Respond with JSON only.
""".strip()

# 2) Tiny helper: pull JSON out of model output even if it adds a code fence
def _extract_json(s: str) -> str:
    # Prefer fenced ```json blocks
    m = re.search(r"```json\s*(\{.*?\})\s*```", s, flags=re.S|re.I)
    if m: 
        return m.group(1)
    # Fallback: first {...} block
    m = re.search(r"(\{.*\})", s, flags=re.S)
    return m.group(1) if m else "{}"

# 3) Per-text extractor using your o3 function
def extract_adme_lists_with_o3(text: str) -> dict:
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return {"absorption": [], "distribution": [], "metabolism": [], "excretion": []}

    prompt = build_adme_prompt(text)
    raw = get_completion_o3(prompt)  # <-- your existing o3 call
    try:
        data = json.loads(_extract_json(raw))
    except Exception:
        data = {}

    # Normalize + defaults
    return {
        "absorption": list(map(str, data.get("absorption", []) or [])),
        "distribution": list(map(str, data.get("distribution", []) or [])),
        "metabolism": list(map(str, data.get("metabolism", []) or [])),
        "excretion": list(map(str, data.get("excretion", []) or [])),
    }

# 4) DataFrame helper: add four list columns from a PK text column
def add_adme_columns(df: pd.DataFrame, pk_text_col: str) -> pd.DataFrame:
    df = df.copy()
    out = df[pk_text_col].apply(extract_adme_lists_with_o3)
    df["adme_absorption"]  = out.apply(lambda d: d["absorption"])
    df["adme_distribution"] = out.apply(lambda d: d["distribution"])
    df["adme_metabolism"]   = out.apply(lambda d: d["metabolism"])
    df["adme_excretion"]    = out.apply(lambda d: d["excretion"])
    return df

import re, json, ast

def _balanced_json_slice(s: str) -> str | None:
    """Return the first balanced {...} slice, handling strings/escapes."""
    start = s.find("{")
    if start == -1:
        return None
    depth, in_str, esc = 0, False, False
    for i in range(start, len(s)):
        ch = s[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return s[start:i+1]
    return None

SECTION_RE = re.compile(
    r"(?is)\b(Absorption|Distribution|Metabolism|Excretion)\s*:\s*(.*?)(?=\n\s*(?:Absorption|Distribution|Metabolism|Excretion)\s*:|\Z)"
)

def _split_sents(block: str):
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z(])', block.strip())
    return [s.strip().rstrip(" .") for s in parts if s.strip()]

def _parse_headed_fallback(text: str):
    out = {"absorption": [], "distribution": [], "metabolism": [], "excretion": []}
    if not isinstance(text, str) or not text.strip():
        return out
    sections = {m.group(1).lower(): m.group(2) for m in SECTION_RE.finditer(text)}
    for k in out.keys():
        if k in sections and sections[k].strip():
            out[k] = _split_sents(sections[k])
    return out

def extract_adme_lists(raw: str) -> dict:
    """
    Robustly parse ADME lists from LLM output.
    Prefers strict JSON:
    {
      "absorption": [...],
      "distribution": [...],
      "metabolism": [...],
      "excretion": [...]
    }
    Falls back to parsing 'Absorption: ...' headed text.
    """
    empty = {"absorption": [], "distribution": [], "metabolism": [], "excretion": []}
    if not isinstance(raw, str) or not raw.strip():
        return empty

    s = raw.strip()

    # 1) fenced ```json block
    m = re.search(r"```json\s*(\{.*?\})\s*```", s, flags=re.S | re.I)
    if not m:
        m = re.search(r"```\s*(\{.*?\})\s*```", s, flags=re.S | re.I)
    json_str = m.group(1) if m else None

    # 2) balanced {...} slice
    if not json_str:
        json_str = _balanced_json_slice(s)

    if json_str:
        # sanitize smart quotes and trailing commas
        js = (json_str
              .replace("\u201c", '"').replace("\u201d", '"')
              .replace("\u2019", "'").replace("\u2018", "'"))
        js = re.sub(r',\s*([}\]])', r'\1', js)

        # Try strict JSON
        try:
            obj = json.loads(js)
            return {
                "absorption": list(map(str, obj.get("absorption", []) or [])),
                "distribution": list(map(str, obj.get("distribution", []) or [])),
                "metabolism": list(map(str, obj.get("metabolism", []) or [])),
                "excretion": list(map(str, obj.get("excretion", []) or [])),
            }
        except Exception:
            # Try Python-literal fallback (handles single quotes/True/False/None after conversion)
            pyish = re.sub(r'\btrue\b', 'True', js, flags=re.I)
            pyish = re.sub(r'\bfalse\b', 'False', pyish, flags=re.I)
            pyish = re.sub(r'\bnull\b', 'None', pyish, flags=re.I)
            try:
                obj = ast.literal_eval(pyish)
                if isinstance(obj, dict):
                    return {
                        "absorption": list(map(str, obj.get("absorption", []) or [])),
                        "distribution": list(map(str, obj.get("distribution", []) or [])),
                        "metabolism": list(map(str, obj.get("metabolism", []) or [])),
                        "excretion": list(map(str, obj.get("excretion", []) or [])),
                    }
            except Exception:
                pass

    # 3) Last resort: parse your “Absorption: …” headed format
    return _parse_headed_fallback(s)


In [10]:
df = pd.read_csv('df_pharmacokinetics.csv')
df_2 = df[['ingredient_name','pharmacokinetics']]
df_3 = df_2.drop_duplicates()
#df_2.shape
#df_3.columns
df_3.shape

(110, 2)

In [11]:
df_3

,ingredient_name,pharmacokinetics
0,Amphetamine / Dextroamphetamine,NaN
1,aripiprazole,12.3 Pharmacokinetics Aripiprazole activity is...
2,clonidine hydrochloride,Pharmacokinetics The pharmacokinetics of cloni...
3,asenapine,12.3 Pharmacokinetics Following a single 5 mg ...
4,bupropion hydrochloride,12.3 Pharmacokinetics Bupropion is a racemic m...
...,...,...
105,triazolam,NaN
106,vilazodone hydrochloride,12.3 Pharmacokinetics Vilazodone activity is d...
107,zaleplon,Pharmacokinetics The pharmacokinetics of zalep...
108,ziprasidone hydrochloride,12.3 Pharmacokinetics Oral Pharmacokinetics Zi...


In [12]:
df_3.columns

Index(['ingredient_name', 'pharmacokinetics'], dtype='object')

In [25]:
##----------------------------## Testing ##---------------------------------##
#Indications = []
for ing,text in zip(df_3['ingredient_name'],df_3['pharmacokinetics']):
    print('---------')
    print(ing)
    #print(text)
    prompt = f'''
        1. You are tasked with extracting ADME - absorption, distribution, metabolism and excretion mentioned in the provided 'pharmacokinetics' section about a drug.
        2. Strictly use the text passed to you to extract the ADME from the text passed to you. Do not use outside information to make up the ADME's.  
        3. Once the ADME is extracted, validate it against the original text to make sure it matches. If it does not match remove that from your response. 
        4. Exclude any pretexts or any system messages
        5. OUTPUT JSON ONLY with this exact schema (no extra text):
            {
              "absorption": ["..."],
              "distribution": ["..."],
              "metabolism": ["..."],
              "excretion": ["..."]
            }
        
        Do the task briefed within triple backticks ```{text}```'''
    response = get_completion_o3(prompt)  
    adme = extract_adme_lists(raw_text)
    print(adme)
    #Indications.append(response)
    print('---------')

---------
Amphetamine / Dextroamphetamine


ValueError: Invalid format specifier ' ["..."],
              "distribution": ["..."],
              "metabolism": ["..."],
              "excretion": ["..."]
            ' for object of type 'str'

In [31]:
import pandas as pd

# assuming you already have: get_completion_o3(prompt) and extract_adme_lists(response)

abs_rows, dist_rows, met_rows, exc_rows = [], [], [], []

for ing, text in zip(df_3['ingredient_name'], df_3['pharmacokinetics']):
    print('---------')
    print(ing)

    # skip empty / NaN pharmacokinetics
    if text is None or (isinstance(text, float) and pd.isna(text)) or not str(text).strip():
        abs_rows.append([]); dist_rows.append([]); met_rows.append([]); exc_rows.append([])
        print("No PK text."); continue

    prompt = f"""
        1. You are tasked with extracting ADME - absorption, distribution, metabolism and excretion mentioned in the provided 'pharmacokinetics' section about a drug.
        2. Strictly use the text passed to you to extract the ADME from the text passed to you. Do not use outside information to make up the ADME's.  
        3. Once the ADME is extracted, validate it against the original text to make sure it matches. If it does not match remove that from your response. 
        4. Exclude any pretexts or any system messages
        5. OUTPUT JSON ONLY, exactly in this schema (no extra text):
        {{
          "absorption": ["..."],
          "distribution": ["..."],
          "metabolism": ["..."],
          "excretion": ["..."]
        }}
        
        Text:
        ```{text}```
"""

    response = get_completion_o3(prompt)        # <-- call o3
    adme = extract_adme_lists(response)         # <-- use response, not raw_text
    print(adme)

    abs_rows.append(adme["absorption"])
    dist_rows.append(adme["distribution"])
    met_rows.append(adme["metabolism"])
    exc_rows.append(adme["excretion"])

    print('---------')

# add columns to df_3
df_4 = df_3.copy()
df_4["adme_absorption"]  = abs_rows
df_4["adme_distribution"] = dist_rows
df_4["adme_metabolism"]   = met_rows
df_4["adme_excretion"]    = exc_rows


---------
Amphetamine / Dextroamphetamine
No PK text.
---------
aripiprazole
{'absorption': ['Aripiprazole is well absorbed after administration of the tablet, with peak plasma concentrations occurring within 3 hours to 5 hours; the absolute oral bioavailability of the tablet formulation is 87%.', 'Aripiprazole can be administered with or without food.', 'Administration of a 15 mg aripiprazole tablet with a standard high-fat meal did not significantly affect the C max or AUC of aripiprazole or its active metabolite, dehydro-aripiprazole, but delayed T max by 3 hours for aripiprazole and 12 hours for dehydro-aripiprazole.'], 'distribution': ['The steady-state volume of distribution of aripiprazole following intravenous administration is high (404 L or 4.9 L/kg), indicating extensive extravascular distribution.', 'At therapeutic concentrations, aripiprazole and its major metabolite are greater than 99% bound to serum proteins, primarily to albumin.', 'In healthy human volunteers administ

In [32]:
df_4

,ingredient_name,pharmacokinetics,adme_absorption,adme_distribution,adme_metabolism,adme_excretion
0,Amphetamine / Dextroamphetamine,NaN,[],[],[],[]
1,aripiprazole,12.3 Pharmacokinetics Aripiprazole activity is...,[Aripiprazole is well absorbed after administr...,[The steady-state volume of distribution of ar...,[Elimination of aripiprazole is mainly through...,[Following a single oral dose of [ 14 C]-label...
2,clonidine hydrochloride,Pharmacokinetics The pharmacokinetics of cloni...,[The absolute bioavailability of clonidine on ...,"[Following intravenous administration, clonidi...",[About 50% of the absorbed dose is metabolized...,[Following oral administration about 40% to 60...
3,asenapine,12.3 Pharmacokinetics Following a single 5 mg ...,"[Following sublingual administration, asenapin...",[Asenapine is rapidly distributed and has a la...,[Direct glucuronidation by UGT1A4 and oxidativ...,[After administration of a single dose of [ 14...
4,bupropion hydrochloride,12.3 Pharmacokinetics Bupropion is a racemic m...,[Following single oral administration of bupro...,[In vitro tests show that bupropion is 84% bou...,[Bupropion is extensively metabolized in human...,[Following oral administration of 200 mg of 14...
...,...,...,...,...,...,...
105,triazolam,NaN,[],[],[],[]
106,vilazodone hydrochloride,12.3 Pharmacokinetics Vilazodone activity is d...,[Vilazodone concentrations peak at a median of...,[Vilazodone is widely distributed and approxim...,[Elimination of vilazodone is primarily by hep...,[with only 1% of the dose recovered in the uri...
107,zaleplon,Pharmacokinetics The pharmacokinetics of zalep...,[Zaleplon is rapidly and almost completely abs...,[Zaleplon is a lipophilic compound with a volu...,"[After oral administration, zaleplon is extens...","[After either oral or IV administration, zalep..."
108,ziprasidone hydrochloride,12.3 Pharmacokinetics Oral Pharmacokinetics Zi...,[Ziprasidone is well absorbed after oral admin...,[Ziprasidone has a mean apparent volume of dis...,[Elimination of ziprasidone is mainly via hepa...,[Approximately 20% of the dose is excreted in ...


In [233]:
#pd.to_csv(df_4_out.columns)

In [52]:
for i in df_4['adme_absorption'][0:4]:
    print('-------------')
    print(i)

-------------
[]
-------------
['Aripiprazole is well absorbed after administration of the tablet, with peak plasma concentrations occurring within 3 hours to 5 hours; the absolute oral bioavailability of the tablet formulation is 87%.', 'Aripiprazole can be administered with or without food.', 'Administration of a 15 mg aripiprazole tablet with a standard high-fat meal did not significantly affect the C max or AUC of aripiprazole or its active metabolite, dehydro-aripiprazole, but delayed T max by 3 hours for aripiprazole and 12 hours for dehydro-aripiprazole.']
-------------
['The absolute bioavailability of clonidine on oral administration is 70% to 80%.', 'Peak plasma clonidine levels are attained in approximately 1 to 3 hours.']
-------------
['Following sublingual administration, asenapine is rapidly absorbed with peak plasma concentrations occurring within 0.5 to 1.5 hours.', 'The absolute bioavailability of sublingual asenapine at 5 mg is 35%.', 'Increasing the dose from 5 mg t

In [53]:
for i in df_4['adme_absorption'][0:4]:
    print('----------------')
    prompt = f"""
        1. You are a senior clinical Pharamacologist, with deep knowledge about drugs development & clinical trials. 
        2. Your task is to extract the quantitative fields from 'Absorption' section extracted from 'pharmacokinetics' section about a drug.
        2. Strictly use the text passed to you to infer the results. Do not use outside information to make up the inferences.  
        3. Once the data fields are extracted, validate it against the original text to make sure it matches. If it does not match remove that from your response. 
        4. Exclude any pretexts or any system messages
        5. OUTPUT JSON ONLY, with Key as the name of the field and value as the value from the text (no extra text):
        {{
          "key": ["value"],
        }}
        
        Text:
        ```{text}```
    """
    response = get_completion_o3(prompt)
    print(i)
    print(response)
    

----------------
[]
{
  "injection_frequency_weeks": ["2", "4"],
  "dose_options_mg": ["150", "210", "300", "405"],
  "steady_state_dose_range_mg_every_2_or_4_weeks": ["150-405"],
  "corresponding_oral_dose_range_mg_daily": ["5-20"],
  "effective_half_life_IM_days": ["30"],
  "half_life_oral_hours": ["30"],
  "dose_correspondence": ["300 mg IM every 2 weeks → 20 mg/day oral", "150 mg IM every 2 weeks → 10 mg/day oral"],
  "time_to_peak_week": ["1"],
  "time_to_steady_state_months": ["3"]
}
----------------
['Aripiprazole is well absorbed after administration of the tablet, with peak plasma concentrations occurring within 3 hours to 5 hours; the absolute oral bioavailability of the tablet formulation is 87%.', 'Aripiprazole can be administered with or without food.', 'Administration of a 15 mg aripiprazole tablet with a standard high-fat meal did not significantly affect the C max or AUC of aripiprazole or its active metabolite, dehydro-aripiprazole, but delayed T max by 3 hours for ari

In [59]:
for i in df_4['adme_absorption'][0:4]:
    print('----------------')
    prompt = f"""
        1. You are a senior clinical Pharamacologist, with deep knowledge about drugs development & clinical trials. 
        2. Your task is to extract the quantitative fields from 'Absorption' section extracted from 'pharmacokinetics' section about a drug.
        3. Extract the following fields - 
            3.1 Peak Plasma concentration or Cmax
            3.2 Oral Bioavailability
            3.3 Absolute Bioavailability
            3.4 Relative Bioavailability
            3.5 Tmax
            3.6 AUC
            3.7 T lag
        4. Strictly use the text passed to you to infer the results. Do not use outside information to make up the inferences.  
        3. Once the data fields are extracted, validate it against the original text to make sure it matches. If it does not match remove that from your response. 
        4. Exclude any pretexts or any system messages
        5. OUTPUT JSON ONLY, with Key as the name of the field and value as the value from the text (no extra text):
        {{
          "Peak Plasma concentration or Cmax": ["peak plasma concentration or cmax value"],
          "Oral Bioavailability": ["Oral Bioavailability value"], 
          "Absolute Bioavailability": ["Absolute Bioavailability value"], 
          "Relative Bioavailability": ["Relative Bioavailability"], 
          "Tmax": ["Tmax"],
          "AUC": ["AUC"], 
          "T lag": ["T lag"]
        }}
        
        Text:
        ```{i}```
    """
    response = get_completion_o3(prompt)
    print(i)
    print(response)

----------------
[]
{}
----------------
['Aripiprazole is well absorbed after administration of the tablet, with peak plasma concentrations occurring within 3 hours to 5 hours; the absolute oral bioavailability of the tablet formulation is 87%.', 'Aripiprazole can be administered with or without food.', 'Administration of a 15 mg aripiprazole tablet with a standard high-fat meal did not significantly affect the C max or AUC of aripiprazole or its active metabolite, dehydro-aripiprazole, but delayed T max by 3 hours for aripiprazole and 12 hours for dehydro-aripiprazole.']
{
  "Peak Plasma concentration or Cmax": [],
  "Oral Bioavailability": ["87%"],
  "Absolute Bioavailability": ["87%"],
  "Relative Bioavailability": [],
  "Tmax": ["3 to 5 hours", "delayed by 3 hours with high-fat meal"],
  "AUC": [],
  "T lag": []
}
----------------
['The absolute bioavailability of clonidine on oral administration is 70% to 80%.', 'Peak plasma clonidine levels are attained in approximately 1 to 3 ho

In [62]:
for i in df_4["adme_absorption"][:4]:
    # If your cell is a list of sentences, join them:
    pk_text = " ".join(i) if isinstance(i, list) else str(i)

    prompt = f"""
        You are extracting quantitative ADME facts from the 'Absorption' section of a drug's Pharmacokinetics.
        
        STRICT RULES
        - Use ONLY the provided text. Do not invent values or use outside info.
        - If a field is not present numerically in the text, output an empty array for that field.
        - Return JSON ONLY (no prose, no code fences), with EXACTLY these keys. Each key maps to an array of objects.
        - Keep units exactly as written in the text. If a range is given, put the numeric range as a two-element list.
        
        SYNONYMS TO MATCH (examples, case-insensitive):
        - Peak Plasma Concentration (Cmax): "Cmax", "C_max", "peak plasma concentration", "maximum concentration".
        - Tmax: "Tmax", "T_max", "time to peak", "time to maximum concentration".
        - AUC: "AUC", "AUC0–∞", "AUC0-∞", "AUC_inf", "AUC0–t", "AUCt", "area under the concentration–time curve".
        - Oral Bioavailability: "oral bioavailability", "F" when clearly for oral route.
        - Absolute Bioavailability: "absolute bioavailability", "F abs", "Fabsolute".
        - Relative Bioavailability: "relative bioavailability", "F rel", "relative to", "vs".
        - Tlag: "Tlag", "lag time".
        
        OUTPUT SCHEMA (JSON ONLY):
        {{
          "Peak Plasma concentration or Cmax": [{{"value": "<verbatim>", "numeric": <number or [low,high] or null>, "unit": "<unit or null>", "analyte": "<parent/metabolite or null>", "condition": "<fed/fasted/single-dose/steady-state or null>", "formulation": "<IR/ER or null>", "dose": "<e.g., 10 mg or null>", "route": "<p.o./i.v. or null>", "notes": "<optional brief note or null>"}}],
          "Oral Bioavailability":              [{{"value": "...", "numeric": ..., "unit": "%", ...}}],
          "Absolute Bioavailability":          [{{"value": "...", "numeric": ..., "unit": "%", ...}}],
          "Relative Bioavailability":          [{{"value": "...", "numeric": ..., "unit": "%", "comparator": "<reference formulation or null>", ...}}],
          "Tmax":                               [{{"value": "...", "numeric": ..., "unit": "h" or "hr" or "min", ...}}],
          "AUC":                                [{{"value": "...", "numeric": ..., "unit": "<e.g., ng·h/mL>", "spec": "<AUC0–∞ / AUC0–t / AUCt / AUCinf>", ...}}],
          "T lag":                              [{{"value": "...", "numeric": ..., "unit": "<e.g., h or min>", ...}}]
        }}
        
        VALIDATION
        - Every record must be directly supported by the text. If uncertain, omit it.
        - If only qualitative info is present (e.g., "high-fat meal does not change AUC"), keep "value" and "notes", set "numeric": null.
        
        TEXT:
        {pk_text}
        """

    response = get_completion_o3(prompt) # your o3 call
    print('------------------')
    print(pk_text)
    print(response)                        # JSON string you can json.loads(...)


------------------

{
  "Peak Plasma concentration or Cmax": [],
  "Oral Bioavailability": [],
  "Absolute Bioavailability": [],
  "Relative Bioavailability": [],
  "Tmax": [],
  "AUC": [],
  "T lag": []
}
------------------
Aripiprazole is well absorbed after administration of the tablet, with peak plasma concentrations occurring within 3 hours to 5 hours; the absolute oral bioavailability of the tablet formulation is 87%. Aripiprazole can be administered with or without food. Administration of a 15 mg aripiprazole tablet with a standard high-fat meal did not significantly affect the C max or AUC of aripiprazole or its active metabolite, dehydro-aripiprazole, but delayed T max by 3 hours for aripiprazole and 12 hours for dehydro-aripiprazole.
{
  "Peak Plasma concentration or Cmax": [],
  "Oral Bioavailability": [],
  "Absolute Bioavailability": [
    {
      "value": "87%",
      "numeric": 87,
      "unit": "%",
      "analyte": "parent",
      "condition": null,
      "formulation"